## Download the data

In [9]:
# %pip install ipywidgets
# %pip install --upgrade pip


  Using cached pip-26.2.2-py3-none-any.whl.metadata (4.6 kB)
Using cached pip-26.2.2-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 26.0.1
    Uninstalling pip-26.0.1:
      Successfully uninstalled pip-26.0.1
Note: you may need to restart the kernel to use updated packages.


In [11]:
import kagglehub

# # Kaggle API token = KGAT_ebf880722b58743f358be8dfefc34645
# kagglehub.login()

# Download latest version
path = kagglehub.competition_download('ensimag-if-2026', output_dir='data')

print("Path to competition files:", path)

100%|██████████| 3.09M/3.09M [00:01<00:00, 2.98MB/s]

Extracting files...
Path to competition files: ./data


### Time Series Cross-Validation Strategy

| Stage | Training Period | Validation / Test Period |
|---|---|---|
| CV Fold 1 | 2020 | Jan–Jun 2021 |
| CV Fold 2 | 2020 – Jun 2021 | Jul–Dec 2021 |
| CV Fold 3 | 2020–2021 | Jan–Jun 2022 |
| CV Fold 4 | 2020 – Jun 2022 | Jul–Dec 2022 |
| **Final Local Holdout** | **2020–2022** | **All of 2023** |
| **Submission Model** | **2020–2023** | **Competition Test (2024)** |

Outcome: train, validation, and cv_split files saved as cv_setup.pkl

In [5]:
import pandas as pd

train = pd.read_csv('data/raw/train.csv', parse_dates=['date'])
test = pd.read_csv('data/raw/test.csv', parse_dates=['date'])


In [9]:
# observations are already from oldest to newest.
# row index is already reset to 0, 1, 2, ...

# Define the four validation periods.
# The start date is INCLUDED; the end date is EXCLUDED.
# For example, the first period covers January through June 2021.
periods = [
    ("2021-01-01", "2021-07-01"),
    ("2021-07-01", "2022-01-01"),
    ("2022-01-01", "2022-07-01"),
    ("2022-07-01", "2023-01-01"),
]


# Keep 2020–2022 for model development and cross-validation.
# .loc selects the rows satisfying those conditions.
dev = train.loc[
    (train["date"] >= "2020-01-01") &
    (train["date"] < "2023-01-01")
].reset_index(drop=True)


# Create an empty list to store the four splits.
# Each split will contain:
#   1. The row positions used for training.
#   2. The row positions used for validation.
cv_splits = []

# Loop through the four validation periods.
for start, end in periods:

    # Training uses ALL development observations before validation starts.
    # This makes the training window expand with each fold.
    train_idx = dev.index[
        dev["date"] < start
    ].to_numpy()

    # Validation uses observations within the current validation period.
    valid_idx = dev.index[
        (dev["date"] >= start) &
        (dev["date"] < end)
    ].to_numpy()

    # Store the pair of arrays in our list.
    # These arrays contain row positions, not the actual data.
    cv_splits.append((train_idx, valid_idx))


# Set aside 2023 for the final evaluation.
# Do not use this data to choose features or tune models.
holdout = train.loc[
    (train["date"] >= "2023-01-01") &
    (train["date"] < "2024-01-01")
].copy()


In [25]:
import pickle

# Save into the existing "data" folder.
with open("data/cv_setup.pkl", "wb") as file:
    pickle.dump(
        {
            "train_data": dev,
            "validation_data": holdout,
            "cv_splits": cv_splits,
        },
        file,
    )


In [ ]:
# import pandas as pd
#
# # Load the dictionary saved in the pickle file.
# setup = pd.read_pickle("data/cv_setup.pkl")
#
# # Extract the two pandas DataFrames and the list of CV splits.
# train_data = setup["train_data"]
# validation_data = setup["validation_data"]
# cv_splits = setup["cv_splits"]